### Tests for ollama

In [ ]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field

True

In [14]:
load_dotenv(override=True)
ollama_api_key = os.getenv('OLLAMA_API_KEY')
print(f"api key: {ollama_api_key}")
ollama_base_url = os.getenv('OLLAMA_BASE_URL')
print(f"base url: {ollama_base_url}")
llama_model_version = os.getenv('OLLAMA_DEFAULT_MODEL_NAME')
print(f"model version: {llama_model_version}")
llama_client = AsyncOpenAI(base_url=ollama_base_url, api_key=ollama_api_key)
llama_model = OpenAIChatCompletionsModel(model=llama_model_version, openai_client=llama_client)
agent1 = Agent(name="ollama Sales Agent", instructions="be honest and concise", model=llama_model)

api key: ollama
base url: http://localhost:11435/v1
model version: llama3.2:1b


In [17]:
result = await Runner.run(agent1, "tell me about yourself")

result.final_output

'I\'m an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."'

In [2]:
load_dotenv(override=True)
default_api_key = os.getenv('DEFAULT_AI_PROVIDER_API_KEY')
print(f"default api key: {default_api_key}")
default_base_url = os.getenv('DEFAULT_AI_PROVIDER_BASE_URL')
print(f"default base url: {default_base_url}")
default_model_version = os.getenv('DEFAULT_MODEL_NAME')
print(f"default model version: {default_model_version}")
default_client = AsyncOpenAI(base_url=default_base_url, api_key=default_api_key)
default_model = OpenAIChatCompletionsModel(model=default_model_version, openai_client=default_client)
default_agent = Agent(name="Default Agent", instructions="be honest and concise", model=default_model)

default api key: ollama
default base url: http://localhost:11435/v1
default model version: qwen2.5:7b


In [4]:
result = await Runner.run(default_agent, "tell me about yourself")

result.final_output

"I'm a large language model developed by Alibaba Cloud. I can generate human-like text to assist with various tasks. How can I help you today?"

### Tests for the deep research agent orchestration project:

In [1]:
import os
from dotenv import load_dotenv
from typing import List
from pydantic import BaseModel
from agents import Agent, ModelSettings, OpenAIChatCompletionsModel, Runner
from openai import AsyncOpenAI

load_dotenv(override=True)
NUM_CLARIFYING_QUESTIONS = 3

def get_default_model():
    """
    Returns the model to use for an Agent based on environment configuration.

    If DEFAULT_AI_PROVIDER is set, builds an OpenAIChatCompletionsModel
    pointed at the custom provider (base URL, API key, model name).
    Otherwise, falls back to the global MODEL_NAME.
    """

    if os.getenv("DEFAULT_AI_PROVIDER") != None:
        agent_client = AsyncOpenAI(
            base_url=os.getenv("DEFAULT_AI_PROVIDER_BASE_URL"),
            api_key=os.getenv("DEFAULT_AI_PROVIDER_API_KEY"),
        )
        model = OpenAIChatCompletionsModel(
            model=os.getenv("DEFAULT_MODEL_NAME"),
            openai_client=agent_client
        )
    else:
        model = os.getenv("DEFAULT_OPENAI_MODEL_NAME", "gpt-5.4-mini")

    return model

# --------------------------------------------------------------------------
# Structured outputs
# --------------------------------------------------------------------------

class ClarifyingQuestions(BaseModel):
    questions: List[str]

model = get_default_model()

INSTRUCTIONS = f"""
A user will give you a question. Before it can be answered well, you need 
more information. Generate exactly {NUM_CLARIFYING_QUESTIONS} short, specific 
clarifying questions that would most help narrow down and best answer the 
user's original question. Do not answer the question yourself. Do not ask 
more or fewer than {NUM_CLARIFYING_QUESTIONS} questions. You must always 
respond in valid JSON format and follow the structured output specified.
"""


clarifying_questions_agent = Agent(
    name="Clarifying Questions Agent",
    model=model,
    instructions=INSTRUCTIONS,
    output_type=ClarifyingQuestions,
)

result = await Runner.run(clarifying_questions_agent, "celebrities who don't eat cheese")
print(f"result:\n{result.final_output}\n")
print(f"count: {len(result.final_output.questions)}\n")

if len(result.final_output.questions) == 0:
    print("No clarification needed!")
else:
    print("Clarification needed!")

APIConnectionError: Connection error.